In [1]:
# Optional install; works from root or provider directory.
from pathlib import Path

_install_root = next(
    p
    for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "notebooks" / "qwen" / "requirements.txt").is_file()
)
_requirements = str(_install_root / "notebooks" / "qwen" / "requirements.txt")
%pip install -r "$_requirements" -q

Note: you may need to restart the kernel to use updated packages.


# Text-to-SQL

# Qwen / Hugging Face analytics with opt-in semantic enforcement

Use the existing SQLite snapshot; do not use Run All for inference-only testing.
Run setup cell 3, imports cell 10, HF configuration cell 12, snapshot/workflow cell 17,
SQL stage cell 19, then Vietnamese response cell 20 (indices are zero-based).
Cells 5, 8 and 15 remain the explicit refresh/schema-loader flow: skip them for benchmarks.

Read [Qwen semantic enforcement guide](SEMANTIC_ENFORCEMENT.md) before running.
Configure HF_TOKEN outside notebook cells in the ignored root .env or environment.
Keep your currently configured HF_MODEL_ID and HF_PROVIDER; do not silently change models.

QWEN_SEMANTIC_MODE defaults to legacy. Select strict for validated semantic plans compiled
to SQL, or shadow for a separate strict comparison without replacing the legacy result.
HF_STRUCTURED_OUTPUT=auto attempts JSON schema with bounded format-incompatibility handling.
Every mode validates JSON locally. A valid JSON shape does not prove correct business meaning.

HF_SQL_REQUEST_TIMEOUT_SECONDS and HF_RESPONSE_REQUEST_TIMEOUT_SECONDS configure separate
HTTP timeouts, falling back to HF_REQUEST_TIMEOUT_SECONDS (30).
HF_MAX_TOKENS (1024) and HF_RESPONSE_MAX_TOKENS (1500) are separate output budgets.
HF_AGENT_MAX_STEPS is capped at 3 total SQL-stage calls, including repairs/reviews/shadow.
HF_FEW_SHOT retains its existing default. BENCHMARK_SQLITE_PATH selects the existing snapshot.
HF_SQL_ALLOWED_TABLES optionally restricts discovered tables. Existing row/byte/SQL timeout
limits remain active. SHOW_SQL_EVIDENCE exposes legacy evidence; strict evidence is visible.

Rerun configuration and workflow initialization after changing settings. Restart the kernel
after dependency/runtime module changes. Snapshot changes invalidate schema/profile caches.
Response validation checks references and values, not every possible semantic contradiction.


In [2]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

REPO_ROOT = next(
    (
        p
        for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (p / "notebooks" / "shared" / "analytics.py").is_file()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the self-healthy-kafka repository.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env")

True

Load the latest local `poc-mssql` snapshot into SQLite:

In [3]:
from datetime import date, datetime, timezone
from decimal import Decimal
from uuid import UUID

import pyodbc
from sqlalchemy import Column, Integer, MetaData, String, Table, Text, create_engine, insert, text


def local_poc_connection_string():
    """Build a local-only connection string without persisting the SA password."""
    import subprocess

    completed = subprocess.run(
        ["docker", "inspect", "poc-mssql", "--format", "{{range .Config.Env}}{{println .}}{{end}}"],
        check=True,
        capture_output=True,
        text=True,
    )
    password_line = next(
        (
            line
            for line in completed.stdout.splitlines()
            if line.startswith(("MSSQL_SA_PASSWORD=", "SA_PASSWORD="))
        ),
        None,
    )
    if not password_line:
        raise RuntimeError("Local poc-mssql password environment variable was not found.")
    password = password_line.split("=", 1)[1]
    return (
        "DRIVER={ODBC Driver 17 for SQL Server};SERVER=localhost,14330;"
        "DATABASE=ingest_reference;UID=sa;PWD=" + password + ";TrustServerCertificate=yes"
    )


snapshot_refresh_metadata = {
    "started_at": datetime.now(timezone.utc).isoformat(),
    "source": "configured MSSQL source",
    "tables": {},
}

source_connection_string = os.getenv("BENCHMARK_MSSQL_CONNECTION_STRING")
if not source_connection_string:
    source_connection_string = local_poc_connection_string()
    print("Using the local poc-mssql snapshot source.")

available_drivers = pyodbc.drivers()
if "ODBC Driver 17 for SQL Server" not in available_drivers:
    raise RuntimeError(
        "ODBC Driver 17 for SQL Server is required for local poc-mssql. "
        f"Installed drivers: {available_drivers}"
    )

snapshot_path = Path(os.getenv("BENCHMARK_SQLITE_PATH", "self_healthy_kafka_snapshot.db"))
if not snapshot_path.is_absolute():
    snapshot_path = REPO_ROOT / snapshot_path
engine = create_engine(f"sqlite:///{snapshot_path}")
metadata_obj = MetaData()

connector_healing_queue = Table(
    "ConnectorHealingQueue",
    metadata_obj,
    Column("QueueId", String(36), primary_key=True),
    Column("RootConnectorName", String(255), nullable=False),
    Column("CurrentConnectorName", String(255), nullable=False),
    Column("ConnectorClass", String(500)),
    Column("HealingMode", String(50), nullable=False),
    Column("QueueStatus", String(50), nullable=False),
    Column("FinalOutcome", String(50)),
    Column("ReceivedAt", String(40), nullable=False),
    Column("StartedAt", String(40)),
    Column("CompletedAt", String(40)),
    Column("NextAttemptAt", String(40)),
    extend_existing=True,
)


def normalize(value):
    if isinstance(value, (UUID, datetime, date, Decimal)):
        return str(value)
    return value


def fetch_rows(cursor, query, columns):
    cursor.execute(query)
    return [
        dict(zip(columns, (normalize(value) for value in row), strict=True))
        for row in cursor.fetchall()
    ]


redact_raw_log_text = os.getenv("BENCHMARK_REDACT_RAW_LOG_TEXT", "true").lower() != "false"
try:
    source_connection = pyodbc.connect(source_connection_string)
except pyodbc.InterfaceError as exc:
    raise RuntimeError(
        "Cannot resolve the ODBC driver in BENCHMARK_MSSQL_CONNECTION_STRING. "
        "Use DRIVER={ODBC Driver 17 for SQL Server}; not Driver 18 or a DSN name. "
        f"Installed drivers: {available_drivers}"
    ) from exc

with source_connection:
    cursor = source_connection.cursor()
    queue_columns = [
        "QueueId",
        "RootConnectorName",
        "CurrentConnectorName",
        "ConnectorClass",
        "HealingMode",
        "QueueStatus",
        "FinalOutcome",
        "ReceivedAt",
        "StartedAt",
        "CompletedAt",
        "NextAttemptAt",
    ]
    snapshot_rows = {
        "ConnectorHealingQueue": fetch_rows(
            cursor,
            """
            SELECT CAST([QueueId] AS varchar(36)), [RootConnectorName], [CurrentConnectorName],
                   [ConnectorClass], [HealingMode], [QueueStatus], [FinalOutcome],
                   CONVERT(varchar(40), [ReceivedAt], 127), CONVERT(varchar(40), [StartedAt], 127),
                   CONVERT(varchar(40), [CompletedAt], 127), CONVERT(varchar(40), [NextAttemptAt], 127)
            FROM [dbo].[ConnectorHealingQueue]
            ORDER BY [ReceivedAt] DESC
        """,
            queue_columns,
        ),
    }

metadata_obj.drop_all(engine, checkfirst=True)
metadata_obj.create_all(engine)
with engine.begin() as connection:
    if snapshot_rows["ConnectorHealingQueue"]:
        connection.execute(insert(connector_healing_queue), snapshot_rows["ConnectorHealingQueue"])

print(
    f"Loaded {len(snapshot_rows['ConnectorHealingQueue'])} latest queue rows into {snapshot_path}."
)
snapshot_refresh_metadata["tables"]["ConnectorHealingQueue"] = {
    "loaded_at": datetime.now(timezone.utc).isoformat(),
    "rows": len(snapshot_rows["ConnectorHealingQueue"]),
}

Using the local poc-mssql snapshot source.
Loaded 6 latest queue rows into C:\Users\ROG STRIX\Documents\GitHub\self-healthy-kafka\self_healthy_kafka_snapshot.db.


### Build our agent

The loader retains its existing physical queue schema. The model context is discovered from the actual snapshot after both tables are available.


In [4]:
# Schema inspection is centralized in the schema-initialization cell below.


Import the reusable parser, read-only executor and generic result/response workflow.


In [ ]:
from notebooks.shared.analytics import Snapshot
from notebooks.shared.semantic_workflow import SemanticWorkflow

Configure one Hugging Face model for both stages. Credentials are read only from the environment; no API call happens in this cell.


In [ ]:
from notebooks.qwen.adapter import QwenClient

hf_token = os.getenv("HF_TOKEN", "").strip()
if not hf_token:
    raise RuntimeError("Set HF_TOKEN in an untracked .env file or environment, then rerun setup.")
hf_model_id = os.getenv("HF_MODEL_ID", "Qwen/Qwen3-4B-Instruct-2507").strip()
hf_provider = os.getenv("HF_PROVIDER", "auto").strip().lower() or "auto"
if not hf_model_id.lower().startswith("qwen/"):
    import warnings
    warnings.warn("Configured HF model is not Qwen; no substitution performed.", stacklevel=2)
hf_client = QwenClient(
    structured_output=os.getenv("HF_STRUCTURED_OUTPUT", "auto").strip().lower(),
    model=hf_model_id,
    provider=hf_provider,
    api_key=hf_token,
    sql_timeout=float(os.getenv("HF_SQL_REQUEST_TIMEOUT_SECONDS", os.getenv("HF_REQUEST_TIMEOUT_SECONDS", "30"))),
    response_timeout=float(os.getenv("HF_RESPONSE_REQUEST_TIMEOUT_SECONDS", os.getenv("HF_REQUEST_TIMEOUT_SECONDS", "30"))),
)

### Level 2: Table joins

Load the second local snapshot table, `ConnectorHealingLogs`, so the original join scenario remains available.

In [7]:
connector_healing_logs = Table(
    "ConnectorHealingLogs",
    metadata_obj,
    Column("Id", String(36), primary_key=True),
    Column("QueueId", String(36), nullable=False),
    Column("ConnectorName", String(255), nullable=False),
    Column("EventType", String(100), nullable=False),
    Column("AttemptNo", Integer),
    Column("HealingStep", Integer),
    Column("Severity", String(50), nullable=False),
    Column("Message", Text),
    Column("Details", Text),
    Column("CreatedAt", String(40), nullable=False),
    extend_existing=True,
)

with pyodbc.connect(source_connection_string) as source_connection:
    cursor = source_connection.cursor()
    log_columns = [
        "Id",
        "QueueId",
        "ConnectorName",
        "EventType",
        "AttemptNo",
        "HealingStep",
        "Severity",
        "Message",
        "Details",
        "CreatedAt",
    ]
    message_expression = "N'[REDACTED]'" if redact_raw_log_text else "[Message]"
    details_expression = "N'[REDACTED]'" if redact_raw_log_text else "[Details]"
    snapshot_rows["ConnectorHealingLogs"] = fetch_rows(
        cursor,
        f"""
        SELECT CAST([Id] AS varchar(36)), CAST([QueueId] AS varchar(36)), [ConnectorName],
               [EventType], [AttemptNo], [HealingStep], [Severity],
               {message_expression}, {details_expression}, CONVERT(varchar(40), [CreatedAt], 127)
        FROM [dbo].[ConnectorHealingLogs]
        ORDER BY [CreatedAt] DESC
    """,
        log_columns,
    )

# Idempotent refresh: clear the previous local snapshot before loading current rows.
metadata_obj.create_all(engine, checkfirst=True)
with engine.begin() as connection:
    connection.execute(text('DELETE FROM "ConnectorHealingLogs"'))
    if snapshot_rows["ConnectorHealingLogs"]:
        connection.execute(insert(connector_healing_logs), snapshot_rows["ConnectorHealingLogs"])

print(f"Loaded {len(snapshot_rows['ConnectorHealingLogs'])} latest log rows.")
snapshot_refresh_metadata["tables"]["ConnectorHealingLogs"] = {
    "loaded_at": datetime.now(timezone.utc).isoformat(),
    "rows": len(snapshot_rows["ConnectorHealingLogs"]),
}
snapshot_refresh_metadata["finished_at"] = datetime.now(timezone.utc).isoformat()

Loaded 16 latest log rows.


Discover schema and initialize a fresh workflow after loading, or against an existing snapshot. Rerun this cell after a refresh or schema/configuration change.


In [ ]:
import json

snapshot_path = Path(os.getenv("BENCHMARK_SQLITE_PATH", "self_healthy_kafka_snapshot.db"))
if not snapshot_path.is_absolute():
    snapshot_path = REPO_ROOT / snapshot_path
allowed = os.getenv("HF_SQL_ALLOWED_TABLES", "").strip()
snapshot = Snapshot(
    snapshot_path,
    allowed_tables=[t.strip() for t in allowed.split(",") if t.strip()] if allowed else None,
    row_limit=int(os.getenv("HF_SQL_RESULT_ROW_LIMIT", "100")),
    byte_limit=int(os.getenv("HF_SQL_RESULT_BYTE_LIMIT", "64000")),
    timeout_seconds=float(os.getenv("HF_SQL_TIMEOUT_SECONDS", "3")),
    refresh_metadata=globals().get("snapshot_refresh_metadata"),
)
semantic_mode = os.getenv("QWEN_SEMANTIC_MODE", "legacy").strip().lower()
workflow = SemanticWorkflow(
    snapshot,
    hf_client,
    mode=semantic_mode,
    model_id=hf_model_id,
    provider=hf_provider,
    max_attempts=int(os.getenv("HF_AGENT_MAX_STEPS", "3")),
    sql_max_tokens=int(os.getenv("HF_MAX_TOKENS", "1024")),
    response_max_tokens=int(os.getenv("HF_RESPONSE_MAX_TOKENS", "1500")),
    few_shot=os.getenv("HF_FEW_SHOT", "true").strip().lower() in {"true", "1", "yes"},
)
print(json.dumps(snapshot.context(), ensure_ascii=False, indent=2))
print("Configuration:", {"model": hf_model_id, "provider": hf_provider, "mode": semantic_mode})

## Two independently timed inference stages

Cell A: edit the question. Legacy asks HF for SQL; strict asks HF for a semantic plan
and executes only compiler-produced SQL. Shadow additionally runs strict when budget remains.
All plan repairs, format-incompatibility handling and reviews share at most 3 SQL-stage calls.
Inspect the plan, assumptions, parameters, SQL, diagnostics and trace before accepting results.

Cell B: ask the same HF model to express verified results in Vietnamese without tools.
Evidence validation failures produce a labeled deterministic table fallback.
For clarification, add the missing metric/denominator to the question and rerun both stages.
Changing the question requires running Cell A before Cell B.


In [ ]:
import json

question = "Thống kê số lượng queue theo từng trạng thái QueueStatus hiện tại."

verified_result = None
final_answer = None
workflow.reset()
try:
    verified_result = workflow.query(question)
    if workflow.clarification:
        print("Clarification required:", workflow.clarification)
    else:
        print("Verified returned rows:", verified_result["returned_row_count"])
        print("Truncated:", verified_result["truncated"])
        print("Model interpretation (not independently verified):", workflow.interpretation)
        if semantic_mode != "legacy" or os.getenv("SHOW_SQL_EVIDENCE", "false").lower() == "true":
            print(json.dumps(verified_result, ensure_ascii=False, indent=2))
finally:
    if semantic_mode != "legacy":
        print("Semantic plan:", json.dumps(workflow.semantic_plan, ensure_ascii=False))
        print("Validation trace:", json.dumps(workflow.trace, ensure_ascii=False))
        if workflow.shadow is not None:
            print("Shadow comparison:", json.dumps(workflow.shadow, ensure_ascii=False))
    print("Step A — Hugging Face + SQLite:", json.dumps(workflow.metrics, ensure_ascii=False))
    if workflow.result is None and not workflow.clarification:
        print("No verified result. Diagnostics:", json.dumps(workflow.trace, ensure_ascii=False))

In [ ]:
import json

final_answer = None
if workflow.question != question:
    raise RuntimeError("Question changed; rerun Cell A before generating a response.")
final_answer = workflow.respond()
print("Response source:", final_answer["source"])
if final_answer.get("reason"):
    print("Fallback reason:", final_answer["reason"])
print(final_answer["text"])
if final_answer.get("scope"):
    print(final_answer["scope"])
print("Step B — Hugging Face response:", json.dumps(workflow.metrics, ensure_ascii=False))